In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
print('hii')

## Renaming files name

In [115]:
# for subdir, dirs, files in os.walk('vehicles'):
#     c1 = 0
#     f1 = 0
#     for file in files:
#         c1 += 1
#         full_path = os.path.join(subdir, file)
#         new = f"{subdir}\{c1:04}.png"
#         os.rename(full_path, new)
#         # print(new)

In [117]:
# Define the "rules" for our images
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [118]:
# Point PyTorch to your folders
train_data = datasets.ImageFolder(root='vehicles', transform=transform)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

In [119]:
train_data.classes

['bikes', 'cars', 'motorcycles', 'planes', 'rickshaws', 'ships', 'trains']

In [120]:
import torch.nn as nn
import torch.nn.functional as F

In [121]:
class VehicleCNN(nn.Module):
  def __init__(self):
    super(VehicleCNN, self).__init__()
    # First layer: 3 input channels (RGB), 16 filters
    self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
    self.pool = nn.MaxPool2d(2, 2) # Reduces size by half
    
    # Second layer: 16 input channels, 32 filters
    self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
    
    # Fully connected layers (The 'Classifier')
    # We must calculate the input size based on the image dimensions
    self.fc1 = nn.Linear(32 * 56 * 56, 128) 
    self.fc2 = nn.Linear(128, 7) # Final output: 7 classes

  def forward(self, x):
    # Sequence: Conv -> ReLU -> Pool
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    
    # Flatten the data for the Linear layers
    x = x.view(-1, 32 * 56 * 56)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

In [122]:
model = VehicleCNN()

In [123]:
import torch.optim as optim

In [125]:
# Use CrossEntropyLoss because we have 7 distinct classes
criterion = nn.CrossEntropyLoss()

# Adam is a popular optimizer that adjusts the learning rate automatically
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Check if a GPU is available to make training faster
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

cpu


VehicleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=100352, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)

In [ ]:
num_epochs = 5
for epoch in range(num_epochs):
  model.train()
  running_loss = 0.0
  
  for images, labels in train_loader:
    # Move data to the same device as the model
    images, labels = images.to(device), labels.to(device)
    
    # --- The 4 Training Steps ---
    optimizer.zero_grad()               # 1. Clear previous gradients
    outputs = model(images)             # 2. Forward pass
    loss = criterion(outputs, labels)   # 3. Calculate loss
    loss.backward()                    # 4. Backward pass
    optimizer.step()                   # 5. Update weights
    
    running_loss += loss.item()
  
  print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

d:\Project\New folder\venv\lib\site-packages\PIL\Image.py:1034: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch [1/10], Loss: 1.7097


KeyboardInterrupt: 

In [60]:
from PIL import Image
import matplotlib.pyplot as plt
import os

In [73]:
def predict_vehicle(image_path):
  # 1. Load and transform the image
  path = os.path.exists(path=image_path)
  if not path:
     print('Path is not found.')
     return
  img = Image.open(image_path).convert('RGB')
  img_tensor = transform(img).unsqueeze(0).to(device) # Add batch dimension
  
  # 2. Set model to evaluation mode
  model.eval()
  
  with torch.no_grad():
      outputs = model(img_tensor)
      # Get probabilities
      probabilities = torch.nn.functional.softmax(outputs, dim=1)
      # Get the index of the highest probability
      conf, predicted_idx = torch.max(probabilities, 1)
  
  class_name = train_data.classes[predicted_idx.item()]
#   plt.imshow(img)
#   plt.text(0, 0, f"{class_name} | {conf.item()*100:.2f}", color="#ccc", fontsize=15, bbox=dict(facecolor='red', alpha=0.8, pad=5))
#   plt.axis('off')
#   plt.show()
  return class_name

In [87]:
predict_vehicle('vehicles/rickshaws/Auto Rickshaw (15).jpg')

'Auto Rickshaws'

## Check accuracy

In [94]:
# Save the model weights
torch.save(model.state_dict(), 'vehicle_model.pth')
print("Model saved successfully!")

Model saved successfully!
